In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from foundry.repository import TrialBalanceRepository
from foundry.config.settings import POSTGRES_TABLE_LOCATIONS

from atlas import AtlasClient

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools, AtlasTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder
from break_analysis.services import AtlasEvidenceService, ReconBreakRecordResolver


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/19 21:34:42 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/19 21:34:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7b93f30d-b0dc-4c5a-8a5d-1f83cd3ad875;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 69ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

atlas_client = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

atlas_evidence_service = AtlasEvidenceService(
    atlas_client=atlas_client, 
    foundry_repository=repository
)

recon_record_resolver = ReconBreakRecordResolver(recon_client=recon)

atlas_tools = AtlasTools(
    recon_resolver=recon_record_resolver,
    atlas_evidence_service=atlas_evidence_service
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    atlas_tools=atlas_tools,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = 'd3282a32-5edc-4d85-b48c-e0ea38d9a4de'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-19 21:34:47,108 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-19 21:34:47,108 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-19 21:34:47,112 | INFO | break_analysis.agent | Analyzing break case | case_id=1d1af593 | workflow_run_id=d3282a32 | topology=MANY_TO_ONE | records=3
2026-09-19 21:34:47,113 | INFO | break_analysis.agent | Invoking LLM | case_id=1d1af593 | round=1
2026-09-19 21:34:59,484 | INFO | break_analysis.agent | LLM invoked | case_id=1d1af593 | round=1 | tool_calls=4 | input_tokens=1265 | output_tokens=203 | total_tokens=1468 | prompt_eval_count=1265 | eval_count=203 | load_ms=4979 | prompt_eval_ms=938 | eval_ms=6437 | total_ms=12363 | duration_ms=12371
2026-09-19 21:34:59,485 | INFO | break_analysis.agent | Tool round | case_id=1d1af593 | round=1 | tool_calls=4
2026-09-19 21:34:59,692 | INFO | break_analysis.agent | Tool invoked | case_id=1d1af593 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "4100000", "business_dt": "2026-03-31"} | duration_ms=207
2026-09-19 21:34:59,810 | INFO | break_analysis.agent | Tool invoked | case_id=1d1af593 | tool=validate_se

BreakAnalysisResult(case_id=UUID('1d1af593-95e1-4378-b8ad-4f53611eb7bb'), recon_result_ids=(UUID('bf31b11b-2740-4649-8543-891f56e01821'), UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), UUID('828d5971-42c9-40dc-b0e7-96e5faf562af')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, findings=(BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), segment_type=<GLSegmentType.ACCOUNT: 'GL_ACCOUNT'>, segment_value='4100000', explanation='Registry validation confirms this GL account is invalid (no active record exists)'), BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), segment_type=<GLSegmentType.SUB_ACCOUNT: 'GL_SUB_ACCOUNT'>, segment_value='004000', explanation='Registry validation confirms this GL sub-account is invalid (no active record exists)'), BreakFinding(root_cause=<RootCause.REGIST

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-19 21:35:36,399 | INFO | break_analysis.agent | Analyzing break case | case_id=7c09ff54 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-19 21:35:36,399 | INFO | break_analysis.agent | Invoking LLM | case_id=7c09ff54 | round=1
2026-09-19 21:35:38,690 | INFO | break_analysis.agent | LLM invoked | case_id=7c09ff54 | round=1 | tool_calls=1 | input_tokens=1015 | output_tokens=50 | total_tokens=1065 | prompt_eval_count=1015 | eval_count=50 | load_ms=117 | prompt_eval_ms=287 | eval_ms=1657 | total_ms=2289 | duration_ms=2290
2026-09-19 21:35:38,690 | INFO | break_analysis.agent | Tool round | case_id=7c09ff54 | round=1 | tool_calls=1
2026-09-19 21:35:38,787 | INFO | break_analysis.agent | Tool invoked | case_id=7c09ff54 | tool=validate_segment | args={"business_dt": "2026-03-31", "segment_type": "GL_ACCOUNT", "segment_value": "210000"} | duration_ms=96
2026-09-19 21:35:38,787 | INFO | break_analysis.agent | Invoking LLM | case_id=7c09ff54 | round=2
2026-09-19 21:35

BreakAnalysisResult(case_id=UUID('7c09ff54-7493-4e76-96ba-eedd9018a776'), recon_result_ids=(UUID('8b947b0d-0d8b-4b7c-925c-2c0af4628151'), UUID('a8887fa1-74d8-46a2-878d-ebf5de37e8db')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, findings=(BreakFinding(root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, recon_result_id=UUID('a8887fa1-74d8-46a2-878d-ebf5de37e8db'), segment_type=<GLSegmentType.ACCOUNT: 'GL_ACCOUNT'>, segment_value='210000', explanation="Registry record exists but is marked as inactive (status='I') for GL_ACCOUNT '210000' as of 2026-03-31."),), explanation="The GL_ACCOUNT '210000' in the investigation record is invalid because its registry record exists but is marked as inactive. This explains the $65,000 discrepancy between interface and GL balances.")

In [9]:
break_case = break_cases[2]

agent.analyze(break_case)

2026-09-19 21:36:00,418 | INFO | break_analysis.agent | Analyzing break case | case_id=be88a5d2 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-19 21:36:00,419 | INFO | break_analysis.agent | Invoking LLM | case_id=be88a5d2 | round=1
2026-09-19 21:36:17,386 | INFO | break_analysis.agent | LLM invoked | case_id=be88a5d2 | round=1 | tool_calls=6 | input_tokens=1019 | output_tokens=450 | total_tokens=1469 | prompt_eval_count=1019 | eval_count=450 | load_ms=124 | prompt_eval_ms=295 | eval_ms=16382 | total_ms=16965 | duration_ms=16967
2026-09-19 21:36:17,387 | INFO | break_analysis.agent | Tool round | case_id=be88a5d2 | round=1 | tool_calls=6
2026-09-19 21:36:17,481 | INFO | break_analysis.agent | Tool invoked | case_id=be88a5d2 | tool=validate_segment | args={"segment_type": "GL_ENTITY", "segment_value": "CAMKTS", "business_dt": "2026-03-31"} | duration_ms=93
2026-09-19 21:36:17,573 | INFO | break_analysis.agent | Tool invoked | case_id=be88a5d2 | tool=validate_segmen

2026-09-19 21:36:20,787 | INFO | break_analysis.agent | Tool invoked | case_id=be88a5d2 | tool=investigate_resolution | args={"workflow_run_id": "d3282a32-5edc-4d85-b48c-e0ea38d9a4de", "recon_result_id": "4424b323-4067-433d-a23f-b534b6cf4a2e", "segment_type": "GL_ACCOUNT"} | duration_ms=3124
2026-09-19 21:36:20,884 | INFO | break_analysis.services.recon | Break record resolved from recon.result | recon_result_id=4424b323 | workflow_run_id=d3282a32
2026-09-19 21:36:21,280 | INFO | break_analysis.services.atlas | Atlas mapping input values retrieved | recon_result_id=4424b323 | mapping_name=ACCOUNT_TB_MAPPING | input_columns=('SRC_APP_CD', 'SRC_ENTITY_CD', 'DATACLASS', 'COA_RULE_ID', 'SRC_ACCOUNT_ID') | distinct_combinations=1
2026-09-19 21:36:21,974 | INFO | break_analysis.agent | Tool invoked | case_id=be88a5d2 | tool=investigate_resolution | args={"recon_result_id": "4424b323-4067-433d-a23f-b534b6cf4a2e", "segment_type": "GL_SUB_ACCOUNT", "workflow_run_id": "d3282a32-5edc-4d85-b48c-e0

BreakAnalysisResult(case_id=UUID('be88a5d2-121d-445c-93fc-2012ddaf8099'), recon_result_ids=(UUID('a0498239-ac79-4748-8ef6-6148cf9151b3'), UUID('4424b323-4067-433d-a23f-b534b6cf4a2e')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, findings=(BreakFinding(root_cause=<RootCause.ATLAS_UNRESOLVED_SEGMENT: 'ATLAS_UNRESOLVED_SEGMENT'>, recon_result_id=UUID('4424b323-4067-433d-a23f-b534b6cf4a2e'), segment_type=<GLSegmentType.ACCOUNT: 'GL_ACCOUNT'>, segment_value='', explanation='Atlas mapping failed to resolve blank GL_ACCOUNT segment. No active candidate found in ACCOUNT_TB_MAPPING for input values.'), BreakFinding(root_cause=<RootCause.ATLAS_UNRESOLVED_SEGMENT: 'ATLAS_UNRESOLVED_SEGMENT'>, recon_result_id=UUID('4424b323-4067-433d-a23f-b534b6cf4a2e'), segment_type=<GLSegmentType.SUB_ACCOUNT: 'GL_SUB_ACCOUNT'>, segment_value='', explanation='Atlas mapping failed to resolve blank GL_SUB_ACCOUNT segment. No active candidate found in ACCOUNT_TB_MAPPING for input values.'), BreakFinding(roo

In [10]:
# spark.stop()